In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install -q langchain langchain-groq langchain-community chromadb pypdf \
                gradio gTTS sentence-transformers rank_bm25 rank_bm25 \
                librosa soundfile openai-whisper

In [ ]:
import os
import gradio as gr
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from gtts import gTTS
import whisper
import time

In [ ]:
# 2. CONFIGURATION
os.environ["GROQ_API_KEY"] = "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"

# Load Whisper for STT
stt_model = whisper.load_model("base")

In [ ]:
# 3. DOCUMENT PROCESSING & HYBRID RETRIEVER SETUP
def initialize_retriever(pdf_path):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    # Dense Retriever (Vector Store)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # Sparse Retriever (BM25)
    bm25_retriever = BM25Retriever.from_documents(splits)
    bm25_retriever.k = 3

    # Hybrid Ensemble
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever], weights=[0.4, 0.6]
    )
    return ensemble_retriever

In [ ]:
# 4. CORE PIPELINE
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1)

def voice_rag_pipeline(audio_path, retriever):
    # Step 1: Speech-to-Text
    result = stt_model.transcribe(audio_path)
    user_text = result['text']
    
    # Step 2: RAG Retrieval & Generation
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
    response_text = qa_chain.run(user_text)
    
    # Step 3: Text-to-Speech
    tts = gTTS(response_text)
    output_audio = "response.mp3"
    tts.save(output_audio)
    
    return user_text, response_text, output_audio

In [ ]:
# 5. GRADIO INTERFACE
def launch_app(pdf_file):
    retriever = initialize_retriever(pdf_file.name)
    
    def process(audio):
        return voice_rag_pipeline(audio, retriever)

    interface = gr.Interface(
        fn=process,
        inputs=gr.Audio(type="filepath"),
        outputs=[gr.Textbox(label="You said"), gr.Textbox(label="Llama-3.1 RAG"), gr.Audio(label="Voice Response")],
        title="Voice-to-Voice Hybrid RAG (Groq + Llama 3.1)",
        description="Upload a PDF and ask questions via voice!"
    )
    interface.launch(debug=True)

launch_app("https://www.rtx.com/collinsaerospace/what-we-do/capabilities")